In [1]:
import os

if not hasattr(os, "_cwd"):
    os.chdir("../")
    os._cwd = True

In [2]:
from minio import Minio
# from pprint import pprint

from rich import print as pprint
from Environment import *

print(TITLE)

print(MINIO_HOST)


class Storage(Minio):
    def __init__(self):
        super().__init__(
            endpoint=MINIO_URL,
            access_key=MINIO_ROOT_USER,
            secret_key=MINIO_ROOT_PASSWORD,
            secure=False,
        )

    def object_exists(self, bucket_name: str, object_name: str) -> bool:
        try:
            self.stat_object(bucket_name, object_name)
            return True
        except Exception:
            return False


s3 = Storage()

pprint(s3.list_buckets())

1Riel Application Product
msl-t470


[Bucket(name='public', creation_date=datetime.datetime(1, 1, 1, 0, 0, tzinfo=datetime.timezone.utc))]

In [3]:
# list data in bucket
bucket_name = "public"

objects = s3.list_objects(bucket_name)
for obj in objects:
    print(obj.object_name)

2026/
assets/


In [6]:
bucket_name = "private"

# create bucket
if not s3.bucket_exists(bucket_name):
    s3.make_bucket(bucket_name)

In [4]:
bucket_name = "public"

# create bucket
if not s3.bucket_exists(bucket_name):
    s3.make_bucket(bucket_name)

In [3]:
bucket_name = "public"

# Step 2: Set bucket policy for public read-only access
policy = f"""{{
    "Version": "2012-10-17",
    "Statement": [
        {{
            "Effect": "Allow",
            "Principal": {{"AWS": ["*"]}},
            "Action": ["s3:GetObject"],
            "Resource": ["arn:aws:s3:::{bucket_name}/*"]
        }},
        {{
            "Effect": "Allow",
            "Principal": {{"AWS": ["*"]}},
            "Action": ["s3:ListBucket"],
            "Resource": ["arn:aws:s3:::{bucket_name}"]
        }}
    ]
}}"""

s3.set_bucket_policy(bucket_name, policy)

In [3]:
# show bucket policy
bucket_name = "public"
pprint(s3.get_bucket_policy(bucket_name))

{"Version":"2012-10-17","Statement":[{"Effect":"Allow","Principal":{"AWS":["*"]},"Action":["s3:GetObject"],"Resourc
e":["arn:aws:s3:::public/*"]},{"Effect":"Allow","Principal":{"AWS":["*"]},"Action":["s3:ListBucket"],"Resource":["a
rn:aws:s3:::public"]}]}

In [4]:
# remove bucket policy to make it private again
bucket_name = "public"
s3.delete_bucket_policy(bucket_name)

In [ ]:
# upload object 'data.jpg' to bucket 'mybucket'
bucket_name = "public"
s3.fput_object(
    bucket_name=bucket_name,
    object_name="data.jpg",
    file_path="data.jpg",
)


# URL: http://gtr-server:9900/public/data.jpg

In [ ]:
# set bucket access policy to public read
bucket_name = "public"

policy = f"""{{
    "Version": "2012-10-17",
    "Statement": [{{
        "Effect": "Allow",
        "Principal": {{"AWS": ["*"]}},
        "Action": ["s3:GetObject"],
        "Resource": ["arn:aws:s3:::{bucket_name}/*"]
    }}]
}}"""

s3.set_bucket_policy(bucket_name, policy)


# upload object 'data.jpg' to bucket 'mybucket'
bucket_name = "public"
s3.fput_object(
    bucket_name=bucket_name,
    object_name="data.jpg",
    file_path="data.jpg",
)


# URL: http://gtr-server:9900/public/data.jpg

---


In [3]:
bucket_name = "private"

# delete every object in the bucket
objects = s3.list_objects(bucket_name, recursive=True)
for obj in objects:
    s3.remove_object(bucket_name, obj.object_name)


# delete bucket
if s3.bucket_exists(bucket_name):
    s3.remove_bucket(bucket_name)

In [ ]:
# remove object 'data.jpg' from bucket 'mybucket'
s3.remove_object(
    bucket_name="mybucket",
    object_name="data.jpg",
)